# 🐿️ Central Park Squirrel Behavior Analysis

**Dataset:** 2018 Central Park Squirrel Census (3,023 observations, 31 columns)
**Tools:** Python, Pandas, NumPy (no visualization libraries, no machine learning)

**Research question:** What patterns can we identify in squirrels' recorded behavioral responses?

**Objective:** Answer this question using exploratory data analysis (counts, proportions and overlaps) on the `Approaches`, `Indifferent` and `Runs from` columns.

> The dataset records observations. It does not establish causation, so no result below should be read as "humans cause squirrels to do X".

## Part 1 — Dataset overview

In [111]:
import numpy as np
import pandas as pd

In [112]:
df = pd.read_csv('data/squirrel_data.csv')
df['Date'] = pd.to_datetime(df['Date'].astype(str), format=('%m%d%Y'))

In [113]:
df.shape

(3023, 31)

In [114]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3023 entries, 0 to 3022
Data columns (total 31 columns):
 #   Column                                      Non-Null Count  Dtype         
---  ------                                      --------------  -----         
 0   X                                           3023 non-null   float64       
 1   Y                                           3023 non-null   float64       
 2   Unique Squirrel ID                          3023 non-null   str           
 3   Hectare                                     3023 non-null   str           
 4   Shift                                       3023 non-null   str           
 5   Date                                        3023 non-null   datetime64[us]
 6   Hectare Squirrel Number                     3023 non-null   int64         
 7   Age                                         2902 non-null   str           
 8   Primary Fur Color                           2968 non-null   str           
 9   Highlight Fur Color

In [115]:
df.columns

Index(['X', 'Y', 'Unique Squirrel ID', 'Hectare', 'Shift', 'Date',
       'Hectare Squirrel Number', 'Age', 'Primary Fur Color',
       'Highlight Fur Color', 'Combination of Primary and Highlight Color',
       'Color notes', 'Location', 'Above Ground Sighter Measurement',
       'Specific Location', 'Running', 'Chasing', 'Climbing', 'Eating',
       'Foraging', 'Other Activities', 'Kuks', 'Quaas', 'Moans', 'Tail flags',
       'Tail twitches', 'Approaches', 'Indifferent', 'Runs from',
       'Other Interactions', 'Lat/Long'],
      dtype='str')

In [116]:
df.dtypes

X                                                    float64
Y                                                    float64
Unique Squirrel ID                                       str
Hectare                                                  str
Shift                                                    str
Date                                          datetime64[us]
Hectare Squirrel Number                                int64
Age                                                      str
Primary Fur Color                                        str
Highlight Fur Color                                      str
Combination of Primary and Highlight Color               str
Color notes                                              str
Location                                                 str
Above Ground Sighter Measurement                         str
Specific Location                                        str
Running                                                 bool
Chasing                 

In [117]:
df.head(5)

,X,Y,Unique Squirrel ID,Hectare,Shift,Date,Hectare Squirrel Number,Age,Primary Fur Color,Highlight Fur Color,...,Kuks,Quaas,Moans,Tail flags,Tail twitches,Approaches,Indifferent,Runs from,Other Interactions,Lat/Long
0,-73.956134,40.794082,37F-PM-1014-03,37F,PM,2018-10-14,3,NaN,NaN,NaN,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9561344937861 40.7940823884086)
1,-73.957044,40.794851,37E-PM-1006-03,37E,PM,2018-10-06,3,Adult,Gray,Cinnamon,...,False,False,False,False,False,False,False,True,me,POINT (-73.9570437717691 40.794850940803904)
2,-73.976831,40.766718,2E-AM-1010-03,02E,AM,2018-10-10,3,Adult,Cinnamon,NaN,...,False,False,False,False,False,False,True,False,NaN,POINT (-73.9768311751004 40.76671780725581)
3,-73.975725,40.769703,5D-PM-1018-05,05D,PM,2018-10-18,5,Juvenile,Gray,NaN,...,False,False,False,False,False,False,False,True,NaN,POINT (-73.9757249834141 40.7697032606755)
4,-73.959313,40.797533,39B-AM-1018-01,39B,AM,2018-10-18,1,NaN,NaN,NaN,...,True,False,False,False,False,False,False,False,NaN,POINT (-73.9593126695714 40.797533370163)


### Column types

- **Numerical:** `X`, `Y`, `Hectare Squirrel Number`
- **Categorical:** `Shift`, `Age`, `Primary Fur Color` (and other text columns such as `Highlight Fur Color` and `Location`)
- **Boolean:** `Running`, `Chasing`, `Climbing`, `Eating`, `Foraging`, `Kuks`, `Quaas`, `Moans`, `Tail flags`, `Tail twitches`, `Approaches`, `Indifferent`, `Runs from`
- **Date-related:** `Date` (stored as an integer in `MMDDYYYY` format)

## Part 2 — Data quality

### Missing values

In [118]:
df.isna().sum()

X                                                0
Y                                                0
Unique Squirrel ID                               0
Hectare                                          0
Shift                                            0
Date                                             0
Hectare Squirrel Number                          0
Age                                            121
Primary Fur Color                               55
Highlight Fur Color                           1086
Combination of Primary and Highlight Color       0
Color notes                                   2841
Location                                        64
Above Ground Sighter Measurement               114
Specific Location                             2547
Running                                          0
Chasing                                          0
Climbing                                         0
Eating                                           0
Foraging                       

In [119]:
df.isna().mean()*100

X                                              0.000000
Y                                              0.000000
Unique Squirrel ID                             0.000000
Hectare                                        0.000000
Shift                                          0.000000
Date                                           0.000000
Hectare Squirrel Number                        0.000000
Age                                            4.002646
Primary Fur Color                              1.819385
Highlight Fur Color                           35.924578
Combination of Primary and Highlight Color     0.000000
Color notes                                   93.979491
Location                                       2.117102
Above Ground Sighter Measurement               3.771088
Specific Location                             84.254052
Running                                        0.000000
Chasing                                        0.000000
Climbing                                       0

Most columns are complete. The largest gaps are in `Color notes` (about 94% missing) and `Highlight Fur Color` (about 36% missing). `Age` (about 4%) and `Primary Fur Color` (about 2%) have smaller gaps.

### Duplicates

In [120]:
df.duplicated().sum()

np.int64(0)

In [121]:
df['Unique Squirrel ID'].duplicated().sum()

np.int64(5)

In [122]:
df[df['Unique Squirrel ID'].duplicated(keep=False)]

,X,Y,Unique Squirrel ID,Hectare,Shift,Date,Hectare Squirrel Number,Age,Primary Fur Color,Highlight Fur Color,...,Kuks,Quaas,Moans,Tail flags,Tail twitches,Approaches,Indifferent,Runs from,Other Interactions,Lat/Long
1,-73.957044,40.794851,37E-PM-1006-03,37E,PM,2018-10-06,3,Adult,Gray,Cinnamon,...,False,False,False,False,False,False,False,True,me,POINT (-73.9570437717691 40.794850940803904)
519,-73.974945,40.772066,7D-PM-1010-01,07D,PM,2018-10-10,1,Adult,Gray,White,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9749446629706 40.7720659696067)
1539,-73.957711,40.798406,40B-AM-1019-06,40B,AM,2018-10-19,6,Juvenile,Gray,White,...,False,False,False,False,False,False,False,True,NaN,POINT (-73.957710975899 40.7984055614047)
2106,-73.977262,40.769282,4C-PM-1010-05,04C,PM,2018-10-10,5,Adult,Gray,Cinnamon,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9772624834791 40.7692821918543)
2167,-73.958236,40.798775,40B-AM-1019-06,40B,AM,2018-10-19,6,Juvenile,Gray,White,...,False,False,False,False,False,False,False,True,NaN,POINT (-73.9582363437135 40.7987752300505)
2234,-73.974852,40.771959,7D-PM-1010-01,07D,PM,2018-10-10,1,Adult,Gray,White,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9748522796455 40.7719586621003)
2271,-73.956967,40.794879,37E-PM-1006-03,37E,PM,2018-10-06,3,Adult,Gray,Cinnamon,...,False,False,False,False,False,False,False,True,me,POINT (-73.9569670900695 40.794879285681304)
2532,-73.976620,40.766191,1F-AM-1010-04,01F,AM,2018-10-10,4,Adult,Gray,NaN,...,False,False,False,False,False,False,True,False,NaN,POINT (-73.9766195697096 40.7661907261547)
2613,-73.976588,40.766095,1F-AM-1010-04,01F,AM,2018-10-10,4,Adult,Gray,NaN,...,False,False,False,False,False,False,True,False,NaN,POINT (-73.9765877920583 40.7660946071574)
2888,-73.977165,40.769305,4C-PM-1010-05,04C,PM,2018-10-10,5,Adult,Gray,Cinnamon,...,False,False,False,False,False,False,False,False,NaN,POINT (-73.9771646835311 40.7693049165084)


There are **no fully duplicated rows**, but **5 `Unique Squirrel ID` values appear more than once**.

### Cleaning decisions

- The original CSV is kept unchanged.
- The three behavior columns (`Approaches`, `Indifferent`, `Runs from`) contain no missing values, so they need no cleaning.
- Columns such as `Age`, `Primary Fur Color` and `Location` could be filled with the mode or an `"Unknown"` label. Columns with more than 10% missing values (such as `Highlight Fur Color` and `Color notes`) should not be filled.
- However, none of these columns are used in this analysis, so **no imputation is applied**. Filling them would add effort without changing the results.

## Part 3 — Behavioral analysis

Focus: `Approaches`, `Indifferent` and `Runs from`.

### Counts of `True` and `False`

In [123]:
columns = ['Approaches', 'Indifferent', 'Runs from']
for column in columns:
    print(df[column].value_counts())

Approaches
False    2845
True      178
Name: count, dtype: int64
Indifferent
False    1569
True     1454
Name: count, dtype: int64
Runs from
False    2345
True      678
Name: count, dtype: int64


### Proportions, ranked from least to most frequent

In [124]:
frequencies_dict = {}

for column in columns:
    frequencies_dict[column] = np.nanmean(df[column]) * 100
frequencies_dict = dict(sorted(frequencies_dict.items(), key=lambda item: item[1]))
for key, value in frequencies_dict.items():
    print(f'{key}: {value:.1f}')

Approaches: 5.9
Runs from: 22.4
Indifferent: 48.1


In [125]:
most = max(frequencies_dict, key=frequencies_dict.get)
least = min(frequencies_dict, key=frequencies_dict.get)
print(f'The most frequently recorded behaviour: {most}: {frequencies_dict[most]:.1f}')
print(f'The least frequently recorded behaviour: {least}: {frequencies_dict[least]:.1f}')

The most frequently recorded behaviour: Indifferent: 48.1
The least frequently recorded behaviour: Approaches: 5.9


### Interpretation

Among all recorded observations:

| Behavior | Share of observations |
|---|---|
| Indifferent | 48.1% |
| Runs from | 22.4% |
| Approaches | 5.9% |

**What this tells us:** indifference is the most frequently recorded response, running away is recorded about four times as often as approaching, and approaching is rare.

**What it cannot tell us:**
- *Why* a squirrel behaved a certain way. The data does not show whether a human was nearby, how close they were, or what they did.
- Anything about individual squirrels. Each row is an observation, not a unique animal.
- That a `False` value means the squirrel never does that behavior, only that it was not recorded.

## Part 4 — Statistical summaries

### Mean of each Boolean column

In [126]:
for column in df.columns:
    if df[column].dtype == 'bool':
        print(f'{column} average: {np.nanmean(df[column]):.2f}')

Running average: 0.24
Chasing average: 0.09
Climbing average: 0.22
Eating average: 0.25
Foraging average: 0.47
Kuks average: 0.03
Quaas average: 0.02
Moans average: 0.00
Tail flags average: 0.05
Tail twitches average: 0.14
Approaches average: 0.06
Indifferent average: 0.48
Runs from average: 0.22


### Verification: the mean equals the proportion of `True` values

In [127]:
for column in df.columns:
    if df[column].dtype == 'bool':
        assert np.isclose(df[column].mean(),(df[column]==True).sum() / len(df))

The assertion passes silently, so the mean of every Boolean column equals its proportion of `True` values.

### Observations recording at least one of the three behaviors

In [128]:
columns = ['Approaches', 'Indifferent', 'Runs from']
df[columns].any(axis=1).sum()

np.int64(2243)

In [129]:
total = df[['Approaches', 'Indifferent', 'Runs from']]
(len(total[total['Indifferent'] == True]) + len(total[total['Runs from'] == True]) + len(total[total['Approaches'] == True]))/len(df)
len(total[total.any(axis=1)])/len(df)

0.741978167383394

### Observations recording multiple behaviors

In [130]:
len(total[total.sum(axis=1) >= 2])

63

### Overlap between the three indicators

In [131]:
a_r = len(total[total[['Approaches', 'Runs from']].all(axis=1)])
a_i = len(total[total[['Approaches', 'Indifferent']].all(axis=1)])
i_r = len(total[total[['Indifferent', 'Runs from']].all(axis=1)])

print(a_r)
print(a_i)
print(i_r)

20
19
32


In [132]:
rate_of_i_r = i_r / (i_r + a_i + a_r)

In [133]:
len(total[total.all(axis=1)])

4

In [134]:
len(total[~total.any(axis=1)])

780

The behaviors are **not mutually exclusive**:

- 2,243 observations (74.2%) record at least one of the three behaviors, and 780 (25.8%) record none.
- 59 observations record exactly two behaviors and 4 record all three.
- Pairwise overlaps: `Indifferent` + `Runs from` = 32, `Approaches` + `Runs from` = 20, `Approaches` + `Indifferent` = 19 (each includes the 4 observations that record all three).
- `Indifferent` and `Runs from` overlap the most, which is surprising because they describe nearly opposite responses.

## Part 5 — Findings

### Research question and objective

**Research question:** What patterns can we identify in squirrels' recorded behavioral responses?

**Objective:** Answer this question using this dataset with the NumPy and Pandas libraries.

### Dataset dimensions and relevant variables

The dataset has 3,023 observations (rows) and 31 columns covering coordinates (float), fur colors (text), activities and interactions (Boolean) and other squirrel attributes. The analysis focuses on three Boolean columns: `Approaches`, `Indifferent` and `Runs from`.

### Data-quality findings

- No fully duplicated rows, but 5 `Unique Squirrel ID` values appear more than once.
- Several columns have substantial missing values, especially `Color notes` (about 94%) and `Highlight Fur Color` (about 36%).
- The three behavior columns are complete, so no cleaning was needed for this analysis.

### Behavioral frequency results

| Behavior | `True` count | Share of observations |
|---|---|---|
| Indifferent | 1,454 | 48.1% |
| Runs from | 678 | 22.4% |
| Approaches | 178 | 5.9% |

Indifference is the most frequently recorded response and approaching is the least frequent.

### Three evidence-based observations

1. **`Indifferent` and `Runs from` overlap the most** (32 observations, versus 20 and 19 for the other pairs). These are nearly opposite behaviors, which points to ambiguity in how observers recorded them.
2. **About 26% of observations (780) record none of the three behaviors.** These behaviors were not recorded for roughly a quarter of the squirrels.
3. **The three proportions sum to about 76%, but only 74.2% of observations record any behavior**, because some observations record more than one.

## Limitations and Possible Sources of Bias

1. **Observation bias:** Squirrel behavior was recorded by human observers, so some behaviors may have been overlooked or interpreted differently.

2. **Limited time period:** The data was collected in 2018, so it may not represent squirrel behavior in other years or seasons.

3. **Limited geographical scope:** The observations come from Central Park, New York, so the findings may not apply to squirrels in other environments.

4. **No causal evidence:** The dataset records behaviors but does not establish what caused them. For example, we cannot conclude that human presence caused squirrels to approach or flee.

5. **Behavioral overlap:** Multiple behaviors may be recorded in the same observation. Therefore, the three interaction indicators should not be treated as mutually exclusive categories.

6. **Interpretation of `False`:** A `False` value indicates that a behavior was not recorded in that observation. It does not necessarily prove that the squirrel never performs that behavior.

7. **Sampling bias:** The dataset includes observed squirrels rather than every squirrel in Central Park. Squirrels that were easier to spot or observe may be overrepresented.

**Conclusion:** These limitations mean our results describe patterns in the recorded observations, not definitive rules about squirrel behavior.